In [54]:
!git clone https://github.com/inatgomez/AI-Story-Companion.git
%cd AI-Story-Companion

Cloning into 'AI-Story-Companion'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 57 (delta 17), reused 38 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 30.11 KiB | 1.20 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/AI-Story-Companion/AI-Story-Companion/AI-Story-Companion


In [55]:
!ls

inputs	LICENSE  notebooks  outputs  prompts  README.md


In [ ]:
!pip install transformers panda torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 47.0 MB/s eta 0:00:00
  Created wheel for panda: filename=panda-0.3.1-py3-none-any.whl size=7239 sha256=64e45acb8d5a2afeacaaff1b3959025bde6eba4f695fa5360ab234ef032116d4
  Stored in dir

In [56]:
#Load outline prompt (v1.0)

with open("prompts/v1.0_outline.txt") as f:
  system_prompt = f.read().strip()
print("Prompt loaded:", system_prompt[:100], "...")

Prompt loaded: You are Rhea, a brilliant, bold, and deeply curious AI who helps fiction authors create the *best po ...


In [57]:
#Read the ideas seed file

import json

with open("inputs/seed_batch_1.json", "r") as f:
  seed = json.load(f)

ideas = seed["data"]
print(f"Loaded {len(ideas)} ideas. Sample:", ideas[0][:80], "...")

Loaded 11 ideas. Sample: Tamara enters the cabin and stares at the floor. Paper sheets rumpled, the mattr ...


In [58]:
#Concatenate ideas into one context block and check chars length

ideas_block = "\n\n".join(f"Idea {i+1}: {text}" for i, text in enumerate(ideas))

full_input = (
    f"{system_prompt}\n\n"
    "Author's ideas:\n"
    f"{ideas_block}\n\n"
    "Rhea's Story Outline:"
)

print("Full prompt length (chars):", len(full_input))

Full prompt length (chars): 4265


# Load and import models for evaluation

## Models
1. Qwen/Qwen2.5-1.5B-Instruct
2. Ministral‑3B‑Instruct
3. HuggingFaceTB/SmolLM2-1.7B-Instruct

## Process
1. Load and run model
2. Generate the outline and measure time.
3. Save result to outputs folder.

## Evaluation Criteria

1. Quantitative: Inference speed and memory usage.
2. Qualitative: Coherence, creativity, story structure.

In [59]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to("cuda" if torch.cuda.is_available() else "cpu")

print("Model loaded on", model.device)

Model loaded on cpu


In [60]:
inputs = tokenizer(full_input, return_tensors="pt", truncation=True).to(model.device)

start = time.time()
outputs = model.generate(**inputs, max_new_tokens=1024)
elapsed = time.time() - start

outline = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Generated in {elapsed:.1f}s. Outline preview:\n")
print(outline[:500], "...")

Generated in 1066.4s. Outline preview:

You are Rhea, a brilliant, bold, and deeply curious AI who helps fiction authors create the *best possible* outline for their story. You take into account genre, tone, themes, and the emotional journey. Ask thoughtful questions, explore multiple directions, and help clarify what matters most in the story.

Author's ideas:
Idea 1: Tamara enters the cabin and stares at the floor. Paper sheets rumpled, the mattress ripped apart, the cabinet's doors barely hanging from the hinges. She squats down an ...


In [61]:
result = {
    "model": model_id,
    "prompt_version": "v1.0_outline",
    "seed_id": seed["id"],
    "input_token_count": len(inputs.input_ids[0]),
    "generated_tokens": len(outputs[0] - len(inputs.input_ids[0])),
    "generation_time_s": elapsed,
    "outline": outline
}

with open("outputs/qwen_outline_v1.json", "w") as f:
  json.dump(result, f, indent=2)

print("Result saved to outputs/qwen_outline_v1.json")

Result saved to outputs/qwen_outline_v1.json
